In [6]:
#####기본 분데이터 크롤링
import time
import datetime
import pandas as pd
import requests

url = 'https://www.binance.com/fapi/v1/continuousKlines?interval=1m&contractType=PERPETUAL&pair=XRPUSDT'
webpage = requests.get(url) #(1) 인터넷을 통해 데이터 받아오기
webpage.content             #(2) 데이터 출력

b'[[1734810600000,"2.2207","2.2286","2.2207","2.2219","785223.9",1734810659999,"1746705.50376",3533,"446182.1","992349.52176","0"],[1734810660000,"2.2219","2.2255","2.2186","2.2197","744091.1",1734810719999,"1652738.24955",3186,"445618.8","989811.28815","0"],[1734810720000,"2.2197","2.2225","2.2165","2.2168","660814.0",1734810779999,"1466428.36351",2973,"362586.7","804606.13231","0"],[1734810780000,"2.2169","2.2273","2.2169","2.2251","828375.4",1734810839999,"1841378.95099",3191,"528766.0","1175254.77823","0"],[1734810840000,"2.2252","2.2300","2.2236","2.2288","748596.0",1734810899999,"1667524.95296",2953,"327025.5","728442.15687","0"],[1734810900000,"2.2288","2.2325","2.2258","2.2263","760459.3",1734810959999,"1695649.20732",3034,"356184.5","794202.10021","0"],[1734810960000,"2.2262","2.2340","2.2262","2.2313","842650.1",1734811019999,"1880377.65275",2782,"483184.7","1078078.19722","0"],[1734811020000,"2.2313","2.2350","2.2308","2.2347","732898.7",1734811079999,"1636859.19267",2666,"5

In [5]:
#####120일 전 데이터 가져오기
base_url = "https://www.binance.com/fapi/v1/klines?symbol=XRPUSDT&interval=1m&limit=1000&startTime={}"
gettimestamp = int(time.time() - 60*60*24 * 120)*1000 #(2) 120일 시간지정
base_url = base_url.format(gettimestamp)               #(3) startTime 항목에 시간 설정
webpage = requests.get(base_url)                       
webpage.content

<class 'bytes'>


In [ ]:
#####데이터프레임으로 전환
df_candle_temp = pd.read_json(webpage.content)
df_candle_temp

In [10]:
#####timestamp 데이터 확인하기

def get_date(mili_time):
    KST = datetime.timezone(datetime.timedelta(hours=9))
    dt = datetime.datetime.fromtimestamp(mili_time / 1000.0, tz=KST)
    timeline = str(dt.strftime('%D %H:%M:%S'))  #(1) 출력형식 지정
    return timeline
get_date(1614450660000)

'02/28/21 03:31:00'

In [31]:
#####120일 전체 데이터 받기

base_url = "https://www.binance.com/fapi/v1/klines?symbol=XRPUSDT"+ \
                "&interval=1m&limit=1000&startTime={}"
gettimestamp = int(time.time() - 60*60*24 * 120)*1000

for i in range(int(120*1.4)): #(1) 데이터 건수 설정 24시간 * 120일
    
    #(2)타임스템프 설정
    url = base_url.format(int(gettimestamp))       
    
    webpage = requests.get(url)
    
    #(3)JSON 형식 데이터 읽어서 임시 데이터프레임에 저장
    df_candle_temp = pd.read_json(webpage.content) 

    #(4) 새로받은 데이터를 기존 데이터프레임과 병합
    df_candle = pd.concat([df_candle,df_candle_temp],axis=0) 
    
    #(5)마지막 타임스템프 추출
    gettimestamp = df_candle_temp[0][-1:].values[0] 
    
#(6)컬럼명 수정
rename_columns = {0: 't', 1: 'o', 2: 'h', 3: 'l', 4: 'c', 5: 'v'}
df_candle = df_candle[[0, 1, 2, 3, 4, 5]].rename(columns=rename_columns)

#(7)널 데이터 삭제
df_candle = df_candle.dropna(axis=0)

#(8)파일로 저장
df_candle.to_csv("./data/XRPUSDT.csv", index=False)

In [12]:
#####최종 데이터 확인

#gettimestamp = df_candle['t'][-1:].values[0]
timestamp = time.time()
get_date(timestamp)

'01/21/70 10:54:07'